# Script 06: Alignment QC Review & Signoff

**Purpose:** Visual quality control and formal sign-off for COMET→STOmics alignments.  
**Gate:** Downstream analysis (`post_alignment_analysis.py`) will refuse to run until all samples are approved here.

## Workflow
1. Scan alignment output directory for all processed samples
2. Show approval status dashboard
3. Generate QC images (global overview + ROI zoom panels)
4. Walk through each pending sample for interactive review
5. Final status summary

**Key features:**
- File fingerprinting (SHA-256) detects if VALIS warp files changed after approval
- Approval is invalidated automatically if any input/output file is modified
- Re-review is required after re-running alignment on any sample

In [ ]:
# ============================================================
# Configuration — edit these paths for your setup
# ============================================================

# Directory containing per-sample alignment output folders
# Each subfolder is named like SO34_A03979E2/ and contains:
#   - *_warped_segmentations.geojson
#   - *_comet_stomics_integrated.h5ad
#   - registration_output/
#   - validation_manifest.json  (created by alignment pipeline)
ALIGNMENT_DIR = r'T:\Sammy Data\projects\out\comet_stomics_alignment'

# Who is reviewing (recorded in the manifest)
REVIEWER = 'Trevor'

# Number of ROI zoom panels per sample
N_ROIS = 4

# ROI half-width in pixels (each panel shows a 2*ROI_SIZE square)
ROI_SIZE = 500

In [ ]:
import sys
from pathlib import Path

# Ensure the repo root is on the path
repo_root = Path('.').resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from comet.alignment_validation import (
    AlignmentValidator,
    get_approval_status,
    require_approval,
    generate_global_qc,
    generate_roi_qc,
)

import matplotlib
matplotlib.use('inline')
import matplotlib.pyplot as plt
%matplotlib inline

from IPython.display import display, Image, HTML, clear_output
import pandas as pd

print('Imports OK')

---
## 1. Status Dashboard

Scan all sample directories and show current approval status.

In [ ]:
alignment_dir = Path(ALIGNMENT_DIR)
assert alignment_dir.exists(), f"Alignment directory not found: {alignment_dir}"

status_df = get_approval_status(alignment_dir)

# Color-code the status for readability
def style_status(val):
    colors = {
        'approved': 'background-color: #c6efce; color: #006100',
        'pending': 'background-color: #ffeb9c; color: #9c5700',
        'rejected': 'background-color: #ffc7ce; color: #9c0006',
        'NO_MANIFEST': 'background-color: #d9d9d9; color: #404040',
    }
    return colors.get(val, '')

def style_files(val):
    if val == 'YES':
        return 'background-color: #c6efce'
    elif val == 'CHANGED':
        return 'background-color: #ffc7ce; font-weight: bold'
    return ''

styled = (status_df.style
    .applymap(style_status, subset=['status'])
    .applymap(style_files, subset=['files_ok'])
)

n_total = len(status_df)
n_approved = (status_df['status'] == 'approved').sum()
n_pending = (status_df['status'] == 'pending').sum()
n_rejected = (status_df['status'] == 'rejected').sum()
n_no_manifest = (status_df['status'] == 'NO_MANIFEST').sum()
n_changed = (status_df['files_ok'] == 'CHANGED').sum()

display(HTML(f"""
<h3>Alignment Approval Dashboard</h3>
<table style="font-size:14px">
<tr><td><b>Total samples:</b></td><td>{n_total}</td></tr>
<tr><td style="color:#006100"><b>Approved:</b></td><td>{n_approved}</td></tr>
<tr><td style="color:#9c5700"><b>Pending review:</b></td><td>{n_pending}</td></tr>
<tr><td style="color:#9c0006"><b>Rejected:</b></td><td>{n_rejected}</td></tr>
<tr><td style="color:#404040"><b>No manifest:</b></td><td>{n_no_manifest}</td></tr>
<tr><td style="color:#9c0006"><b>Files changed (approval invalid):</b></td><td>{n_changed}</td></tr>
</table>
"""))

display(styled)

---
## 2. Generate QC Images

For each sample that is **pending** or has **changed files**, generate fresh QC visualizations.  
Already-approved samples with unchanged files are skipped.

In [ ]:
# Identify samples that need QC generation
needs_qc = status_df[
    (status_df['status'].isin(['pending', 'rejected'])) |
    (status_df['files_ok'] == 'CHANGED')
]['directory'].tolist()

print(f"Generating QC images for {len(needs_qc)} sample(s)...")
print(f"Samples: {needs_qc}")

qc_results = {}

for i, dirname in enumerate(needs_qc):
    sample_dir = alignment_dir / dirname
    sample_id = dirname.split('_')[0]
    print(f"\n[{i+1}/{len(needs_qc)}] {dirname}...")
    
    try:
        validator = AlignmentValidator(sample_dir)
        global_path, roi_path = validator.generate_qc(
            n_rois=N_ROIS,
            roi_size=ROI_SIZE,
        )
        qc_results[dirname] = {
            'global': global_path,
            'roi': roi_path,
            'status': 'ok',
        }
        print(f"  Global QC: {global_path}")
        print(f"  ROI QC:    {roi_path}")
    except Exception as e:
        qc_results[dirname] = {'status': 'error', 'error': str(e)}
        print(f"  ERROR: {e}")

print(f"\nDone. {sum(1 for v in qc_results.values() if v['status']=='ok')}/{len(needs_qc)} succeeded.")

---
## 3. Interactive Review & Signoff

Walk through each pending/needs-review sample one at a time.  
For each sample:
- Shows the global overview (DAPI blend, outlines, mask coverage, distance histogram)
- Shows ROI zoom panels (phenotype-colored cell boundaries)
- Shows file integrity check
- Prompts for approval: `y` / `n` / `skip` / free-text rejection reason

**Tip:** If a sample looks bad, type a specific reason (e.g., "rotation off in bottom-right") — this gets saved to the manifest for whoever re-runs alignment.

In [ ]:
# Interactive signoff loop
reviewed = []

for dirname in needs_qc:
    sample_dir = alignment_dir / dirname
    sample_id = dirname.split('_')[0]
    
    print(f"\n{'='*70}")
    print(f" REVIEW: {dirname}")
    print(f"{'='*70}")
    
    validator = AlignmentValidator(sample_dir)
    
    # Show QC images
    global_qc = sample_dir / f'{sample_id}_alignment_qc_global.png'
    roi_qc = sample_dir / f'{sample_id}_alignment_qc_rois.png'
    
    if global_qc.exists():
        display(HTML(f'<h3>Global QC: {sample_id}</h3>'))
        display(Image(filename=str(global_qc), width=950))
    else:
        print(f"  [!] No global QC image. Run the QC generation cell above first.")
    
    if roi_qc.exists():
        display(HTML(f'<h3>ROI QC: {sample_id}</h3>'))
        display(Image(filename=str(roi_qc), width=950))
    
    # File integrity
    checks = validator.verify_files()
    if checks:
        display(HTML('<h4>File Integrity</h4>'))
        for role, ok in checks.items():
            icon = '✓' if ok else '✗ CHANGED'
            color = 'green' if ok else 'red'
            display(HTML(f'<span style="color:{color}; font-size:14px">{icon} {role}</span>'))
    
    # Alignment metrics (if available)
    if validator.manifest and 'extra_metadata' in validator.manifest:
        metrics = validator.manifest['extra_metadata'].get('alignment_metrics', {})
        if metrics:
            display(HTML(f"""
            <h4>Alignment Metrics</h4>
            <table style="font-size:13px">
            <tr><td>Quality:</td><td><b>{metrics.get('quality', '?')}</b></td></tr>
            <tr><td>Median distance:</td><td>{metrics.get('median_distance', '?'):.1f} px</td></tr>
            <tr><td>Within 30px:</td><td>{metrics.get('pct_within_30px', '?'):.1f}%</td></tr>
            <tr><td>Within 50px:</td><td>{metrics.get('pct_within_50px', '?'):.1f}%</td></tr>
            </table>
            """))
    
    # Prompt
    print(f'\nOptions: [y]es approve  |  [n]o reject  |  [s]kip  |  <free text> = reject with reason')
    response = input(f'>>> Approve {sample_id}? ').strip()
    
    if response.lower() in ('y', 'yes', 'approve'):
        validator.sign_off(approved=True, reviewer=REVIEWER)
        reviewed.append((dirname, 'APPROVED'))
        print(f'  ✓ APPROVED by {REVIEWER}')
    elif response.lower() in ('s', 'skip', ''):
        reviewed.append((dirname, 'SKIPPED'))
        print(f'  → Skipped (still pending)')
    else:
        reason = response if response.lower() not in ('n', 'no') else 'Rejected by reviewer'
        validator.sign_off(approved=False, reviewer=REVIEWER, reason=reason)
        reviewed.append((dirname, f'REJECTED: {reason}'))
        print(f'  ✗ REJECTED: {reason}')
    
    print()

# Summary
print(f"\n{'='*70}")
print(f" Review Session Summary")
print(f"{'='*70}")
for dirname, result in reviewed:
    print(f"  {dirname}: {result}")

---
## 4. Final Status

Refresh the dashboard to confirm all samples are approved.

In [ ]:
# Refresh status after signoff
status_df = get_approval_status(alignment_dir)

n_approved = (status_df['status'] == 'approved').sum()
n_total = len(status_df)
all_clear = (n_approved == n_total)

if all_clear:
    display(HTML(f"""
    <div style="background:#c6efce; padding:15px; border-radius:8px; margin:10px 0">
        <h3 style="color:#006100; margin:0">✓ All {n_total} samples approved</h3>
        <p style="color:#006100; margin:5px 0 0">You can now run <code>run_post_alignment_pipeline()</code></p>
    </div>
    """))
else:
    n_remaining = n_total - n_approved
    display(HTML(f"""
    <div style="background:#ffeb9c; padding:15px; border-radius:8px; margin:10px 0">
        <h3 style="color:#9c5700; margin:0">⚠ {n_remaining} sample(s) still need review</h3>
        <p style="color:#9c5700; margin:5px 0 0">Re-run the signoff cell above, or address rejections first.</p>
    </div>
    """))

styled = (status_df.style
    .applymap(style_status, subset=['status'])
    .applymap(style_files, subset=['files_ok'])
)
display(styled)

---
## 5. Quick Single-Sample Review (Optional)

If you need to re-review a specific sample (e.g., after re-running VALIS), use this cell.

In [ ]:
# Single sample review
SINGLE_SAMPLE_DIR = None  # e.g., r'T:\Sammy Data\projects\out\comet_stomics_alignment\SO34_A03979E2'

if SINGLE_SAMPLE_DIR:
    val = AlignmentValidator(SINGLE_SAMPLE_DIR)
    
    # Regenerate QC (in case files changed)
    val.generate_qc(n_rois=N_ROIS, roi_size=ROI_SIZE)
    
    # Interactive review
    val.interactive_signoff(reviewer=REVIEWER)

---
## 6. Proceed to Post-Alignment Analysis

Once all samples are approved, run the downstream pipeline.  
This cell will fail with a clear error if any sample is unapproved.

In [ ]:
from comet.post_alignment_analysis import run_post_alignment_pipeline

OUTPUT_DIR = r'T:\Sammy Data\projects\out\cross_sample_analysis'

# This will check approval gates before proceeding
combined, de_results = run_post_alignment_pipeline(
    h5ad_dir=ALIGNMENT_DIR,
    output_dir=OUTPUT_DIR,
    disease_comparison=('Endo', 'MLA'),
    batch_correction_method='harmony',
    stratify_de=True,
)

print(f"Combined data shape: {combined.shape}")
print(f"DE results: {len(de_results)} phenotype strata")